In [1]:
%load_ext autoreload
%autoreload 2

import os
import csv
import warnings
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

#%matplotlib qt

In [ ]:
# --- 1. SECCIÓN DE CONFIGURACIÓN ---
# Modifica estas variables para experimentar con el resultado final.

# Rutas de archivos y carpetas
RUTA_EXCEL = '/home/samuelmr/Documentos/Visual Reasoning/Preguntashumanos.xlsx'
COLUMNA_IMAGEN = 0 # Nombre de la columna con los nombres de archivo
COLUMNA_FRASE = 1        # Nombre de la columna con las frases
RESPUESTA_CORRECTA = 2
TIPO_PREGUNTA = 3

dic_respuesta = ['1 = No    2 = Si', '0 - 4 = Cantidad', '1 = Pequeño    2 = Grande', '1 = Goma    2 = Metal']
CARPETA_IMAGENES_ORIGINALES = '/home/samuelmr/Documentos/Visual Reasoning/CLEVR/img_val/'
CARPETA_SALIDA = '/home/samuelmr/Documentos/Visual Reasoning/img_question/img_val/'

# Propiedades de la nueva imagen
ANCHO_NUEVA_IMAGEN = 1920
ALTO_NUEVA_IMAGEN = 1080
# Color de fondo en formato RGB (R, G, B). Negro: (0, 0, 0), Gris: (40, 40, 40)
COLOR_FONDO = (173, 173, 173) 

FACTOR_ESCALA_IMAGEN = 2

# Propiedades del texto
# Asegúrate de tener el archivo .ttf en una carpeta de fuentes.
RUTA_FUENTE = os.path.join('/home/samuelmr/Documentos/Visual Reasoning/fuentes/', 'Roboto-Regular.ttf')
TAMANO_FUENTE = 40 # 
COLOR_FUENTE = (0, 0, 0)
MARGEN_TEXTO_IMAGEN = 50 # Píxeles desde el borde superior de la imagen

if not os.path.exists(CARPETA_SALIDA):
    os.makedirs(CARPETA_SALIDA)
    print(f" Carpeta de salida creada en: {CARPETA_SALIDA}")

# Cargar el archivo Excel
try:
    df = pd.read_excel(RUTA_EXCEL,  header=None)
except FileNotFoundError:
    print(f" Error: No se encontró el archivo Excel en '{RUTA_EXCEL}'.")
    
except Exception as e:
    print(f" Error al leer el archivo Excel: {e}")

# Iterar sobre cada fila del DataFrame
cont =0
maxlen = 0
contador = 0
lasttext = ''
lista_nombres_generados = []

for index, fila in df.iterrows():
    nombre_imagen = fila[COLUMNA_IMAGEN]
    frase = fila[COLUMNA_FRASE]
    respuestas = fila[TIPO_PREGUNTA]
    cont = cont +1
    # if cont > 16:
    #     continue

    # if len(frase) <= maxlen:
    #     continue
    # else:
    #     print(len(frase))
    #     maxlen=len(frase)
    
    if pd.notna(nombre_imagen) and pd.notna(frase):
        #try:
            aux = 0
            if len(frase) > 70:
                aux=15
                mitad = len(frase) // 2
                try:
                    # Busca el espacio más cercano a la mitad para un corte limpio
                    punto_corte = frase.rfind(' ', 0, mitad)
                    if punto_corte == -1:  # Si no hay espacio antes, busca después
                        punto_corte = frase.find(' ', mitad)
                    if punto_corte != -1:
                        # Reconstruye la frase con un salto de línea
                        frase = frase[:punto_corte] + '\n' + frase[punto_corte + 1:]
                except:
                    # Si algo falla, simplemente corta por la mitad (menos elegante)
                    frase = frase[:mitad] + '\n' + frase[mitad:]
            # if len(frase) == 70:
            #     print(nombre_imagen+'  '+frase)

            frase = frase + '\n\n' + dic_respuesta[int(respuestas/10-1)]

            # Cargar la imagen original
            nombre_imagen = nombre_imagen.replace("test", "val")
            ruta_completa_original = os.path.join(CARPETA_IMAGENES_ORIGINALES, nombre_imagen)
            if not os.path.exists(ruta_completa_original):
                print(f" Advertencia: No se encontró la imagen '{ruta_completa_original}'. Saltando.")

            img_original = Image.open(ruta_completa_original)
            nuevo_ancho = int(img_original.width * FACTOR_ESCALA_IMAGEN)
            nuevo_alto = int(img_original.height * FACTOR_ESCALA_IMAGEN)
            img_redimensionada = img_original.resize((nuevo_ancho, nuevo_alto), Image.Resampling.LANCZOS)

            # Crear la nueva imagen de fondo
            img_nueva = Image.new('RGB', (ANCHO_NUEVA_IMAGEN, ALTO_NUEVA_IMAGEN), color=COLOR_FONDO)
            
            # Calcular la posición para centrar la imagen original
            pos_x_img = (ANCHO_NUEVA_IMAGEN - img_redimensionada.width) // 2
            pos_y_img = (ALTO_NUEVA_IMAGEN - img_redimensionada.height) // 2

            # Pegar la imagen img_redimensionada en la nueva
            img_nueva.paste(img_redimensionada, (pos_x_img, pos_y_img))
            
            # Preparar para dibujar el texto
            draw = ImageDraw.Draw(img_nueva)

            # Intenta cargar una fuente del sistema que sí permite cambiar tamaño.
            fuente = ImageFont.truetype(RUTA_FUENTE, TAMANO_FUENTE)
            
            # Calcular la posición del texto para centrarlo horizontalmente
            bbox = draw.textbbox((0, 0), frase, font=fuente)
            ancho_texto = bbox[2] - bbox[0]
            alto_texto = bbox[3] - bbox[1]

            pos_x_texto = (ANCHO_NUEVA_IMAGEN - ancho_texto) // 2
            pos_y_texto = pos_y_img - alto_texto - MARGEN_TEXTO_IMAGEN + aux
            
            # Dibujar el texto en la imagen
            draw.text((pos_x_texto, pos_y_texto), frase,  fill=COLOR_FUENTE, font=fuente,align='center')

            if nombre_imagen== lasttext:
                contador=contador+1
            else:
                contador=0
            
            # Guardar la imagen resultante
            sufijo_numerico = f"{contador:03d}"
            nombre_base, extension = os.path.splitext(nombre_imagen)
            nuevo_nombre = f"{nombre_base[-4:]}_{sufijo_numerico}{extension}"
            ruta_salida = os.path.join(CARPETA_SALIDA, f"{nuevo_nombre}")
            img_nueva.save(ruta_salida)

            lasttext = nombre_imagen

            lista_nombres_generados.append(nuevo_nombre)
            
        #except Exception as e:
        #    print(f" Ocurrió un error al procesar la imagen {nombre_imagen}")
    else:
        print(f" Advertencia: Fila {index + 2} con datos vacíos. Saltando.")

df_salida = pd.DataFrame(lista_nombres_generados, columns=['nombre_imagen_generada'])
df_salida.to_csv(os.path.join(CARPETA_SALIDA, 'img_names.csv'), index=False, encoding='utf-8')



480
220
480
220
480
220
480
220
480
220
480
220


In [59]:
from eyelinkio import read_edf
import pandas as pd

fname = '/home/samuelmr/Documentos/Lab/Neurosistemas/Christ Devia/nn_eye/data/MU6825/MU6825.edf'
#edf_file = read_edf(fname)

print(edf_file.keys())
# print(edf_file['discrete'].keys())

messages_df = edf_file['discrete']['messages']
print(len(messages_df))
count = 0
for mess in messages_df:
    if 'ima' in str(mess[1]):
        print(str(mess[0]))
        print(str(mess[1]))

# dfs = edf_file.to_pandas()
# print(type(dfs))

# print(dfs)

dict_keys(['info', 'discrete', 'times', 'samples'])
2609
9.120131944444443
b'!V TRIAL_VAR image_sti 0016_002.png'
16.260125874125876
b'!V TRIAL_VAR image_sti 0004_002.png'
28.736137931034484
b'!V TRIAL_VAR image_sti 0040_000.png'
38.11214482758621
b'!V TRIAL_VAR image_sti 0057_000.png'
87.74612499999999
b'!V TRIAL_VAR image_sti 0004_003.png'
104.34828025477707
b'!V TRIAL_VAR image_sti 0028_002.png'
126.0281379310345
b'!V TRIAL_VAR image_sti 0026_001.png'
138.15013194444444
b'!V TRIAL_VAR image_sti 0048_000.png'
147.08813194444443
b'!V TRIAL_VAR image_sti 0000_000.png'
160.97813194444444
b'!V TRIAL_VAR image_sti 0044_001.png'
184.27013194444444
b'!V TRIAL_VAR image_sti 0008_000.png'
203.80014482758622
b'!V TRIAL_VAR image_sti 0011_001.png'
213.62413194444443
b'!V TRIAL_VAR image_sti 0024_000.png'
225.26613793103448
b'!V TRIAL_VAR image_sti 0044_000.png'
236.68413194444443
b'!V TRIAL_VAR image_sti 0038_000.png'
247.99038323353295
b'!V TRIAL_VAR image_sti 0032_001.png'
269.8201319444444
b

In [4]:
#Generador de respuestas correctas

# Rutas de archivos y carpetas
RUTA_EXCEL = 'Preguntashumanos.xlsx'
COLUMNA_IMAGEN = 0      # Nombre de la columna con los nombres de archivo
COLUMNA_FRASE = 1       # Nombre de la columna con las frases
RESPUESTA_CORRECTA = 2
TIPO_PREGUNTA = 3

correct_answers = 'correct_answers.csv'

# Cargar el archivo Excel
try:
    df = pd.read_excel(RUTA_EXCEL,  header=None)
except FileNotFoundError:
    print(f" Error: No se encontró el archivo Excel en '{RUTA_EXCEL}'.")
    
except Exception as e:
    print(f" Error al leer el archivo Excel: {e}")

# Iterar sobre cada fila del DataFrame
maxlen = 0
contador = 0
lasttext = ''
lista_nombres_generados = []

with open(correct_answers, 'w', newline='', encoding='utf-8') as archivo_csv:
    escritor_csv = csv.writer(archivo_csv)
    encabezado = ['img_name', 'correct_answer', 'type_of_question', 'long_question','words']
    escritor_csv.writerow(encabezado)

    for index, fila in df.iterrows():
        nombre_imagen = fila[COLUMNA_IMAGEN]
        frase = fila[COLUMNA_FRASE]
        respuesta_correcta = fila[RESPUESTA_CORRECTA]
        tipo_pregunta = fila[TIPO_PREGUNTA]
        print(str(nombre_imagen)+'\t'+str(respuesta_correcta)+'\t'+str(tipo_pregunta))

        if nombre_imagen== lasttext:
            contador=contador+1
        else:
            contador=0

        sufijo_numerico = f"{contador:03d}"
        nombre_base, extension = os.path.splitext(nombre_imagen)
        nuevo_nombre = f"{nombre_base[-4:]}_{sufijo_numerico}{extension}"

        if tipo_pregunta in [30,40]:
            respuesta_correcta = respuesta_correcta-int(tipo_pregunta/5-4)
        print(nuevo_nombre+'\t\t'+str(respuesta_correcta)+'\t'+str(int(tipo_pregunta/10-1)))

        lista_de_palabras = frase.split()

        fila = [nuevo_nombre, respuesta_correcta, int(tipo_pregunta/10-1), len(frase), len(lista_de_palabras)]
        escritor_csv.writerow(fila)

        lasttext = nombre_imagen


CLEVR_test_000000.png	1	10
0000_000.png		1	0
CLEVR_test_000000.png	0	20
0000_001.png		0	1
CLEVR_test_000000.png	3	30
0000_002.png		1	2
CLEVR_test_000000.png	5	40
0000_003.png		1	3
CLEVR_test_000001.png	1	10
0001_000.png		1	0
CLEVR_test_000001.png	1	20
0001_001.png		1	1
CLEVR_test_000001.png	3	30
0001_002.png		1	2
CLEVR_test_000001.png	5	40
0001_003.png		1	3
CLEVR_test_000002.png	2	10
0002_000.png		2	0
CLEVR_test_000002.png	3	20
0002_001.png		3	1
CLEVR_test_000002.png	3	30
0002_002.png		1	2
CLEVR_test_000002.png	5	40
0002_003.png		1	3
CLEVR_test_000003.png	1	10
0003_000.png		1	0
CLEVR_test_000003.png	2	20
0003_001.png		2	1
CLEVR_test_000003.png	3	30
0003_002.png		1	2
CLEVR_test_000003.png	6	40
0003_003.png		2	3
CLEVR_test_000004.png	2	10
0004_000.png		2	0
CLEVR_test_000004.png	4	20
0004_001.png		4	1
CLEVR_test_000004.png	3	30
0004_002.png		1	2
CLEVR_test_000004.png	6	40
0004_003.png		2	3
CLEVR_test_000005.png	2	10
0005_000.png		2	0
CLEVR_test_000005.png	1	20
0005_001.png		1	1
CLEVR_test